# Загрузка очищенных данных и выделение сегментов

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.tree import DecisionTreeClassifier, export_text

df_clean = pd.read_parquet("data/clean.parquet")
df_clean["is_top"] = ((df_clean["PerformanceRating"] == 4) | (df_clean["JobLevel"] >= 3)).astype(int)
df_clean["Segment"] = df_clean["is_top"].map({1: "Top Talent", 0: "Core Staff"})

top_df = df_clean[df_clean["is_top"] == 1].copy()
df_clean.shape

(1470, 36)

# Статистический отбор признаков (p-value < 0.05)

In [2]:
exclude_cols = {"is_attrition", "is_top", "EmployeeNumber"}
candidate_features = [
    c for c in top_df.select_dtypes(include=[np.number]).columns
    if c not in exclude_cols
]

sig_features = []
for col in candidate_features:
    if top_df[col].nunique() <= 1:
        continue
    _, pval = spearmanr(top_df[col], top_df["is_attrition"])
    if pval < 0.05:
        sig_features.append(col)

sig_features

['Age',
 'DistanceFromHome',
 'EnvironmentSatisfaction',
 'JobInvolvement',
 'JobLevel',
 'JobSatisfaction',
 'MonthlyIncome',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsWithCurrManager',
 'is_overtime']

# Обучение решающего дерева и оценка важности признаков

In [3]:
tree = DecisionTreeClassifier(max_depth=3, min_samples_leaf=30, random_state=42)
tree.fit(top_df[sig_features], top_df["is_attrition"])

importances = pd.Series(tree.feature_importances_, index=sig_features)
importances[importances > 0].sort_values(ascending=False).round(3)

StockOptionLevel    0.337
JobLevel            0.325
MonthlyIncome       0.155
DistanceFromHome    0.138
JobInvolvement      0.045
dtype: float64

In [4]:
print(export_text(tree, feature_names=sig_features))

|--- JobLevel <= 1.50
|   |--- StockOptionLevel <= 0.50
|   |   |--- class: 1
|   |--- StockOptionLevel >  0.50
|   |   |--- class: 0
|--- JobLevel >  1.50
|   |--- DistanceFromHome <= 8.50
|   |   |--- JobInvolvement <= 2.50
|   |   |   |--- class: 0
|   |   |--- JobInvolvement >  2.50
|   |   |   |--- class: 0
|   |--- DistanceFromHome >  8.50
|   |   |--- MonthlyIncome <= 13792.00
|   |   |   |--- class: 0
|   |   |--- MonthlyIncome >  13792.00
|   |   |   |--- class: 0


# Расчет вероятности оттока и категоризация рисков

In [5]:
df_clean["risk_proba"] = tree.predict_proba(df_clean[sig_features])[:, 1]

base_top_rate = top_df["is_attrition"].mean()
df_clean["Risk_Level"] = np.select(
    [
        df_clean["risk_proba"] >= base_top_rate * 2,
        df_clean["risk_proba"] >= base_top_rate
    ],
    [
        "3. High",
        "2. Medium"
    ],
    default="1. Low"
)

df_clean["Risk_Level"].value_counts()

Risk_Level
1. Low       869
2. Medium    344
3. High      257
Name: count, dtype: int64

# Валидация модели на сегменте Top Talent

In [6]:
val = df_clean[df_clean["is_top"] == 1].groupby("Risk_Level").agg(
    total_top=("is_attrition", "count"),
    left_company=("is_attrition", "sum"),
    active_lookalikes=("is_attrition", lambda x: (x == 0).sum()),
    attrition_rate=("is_attrition", lambda x: round(x.mean() * 100, 1))
)
val

,total_top,left_company,active_lookalikes,attrition_rate
Risk_Level,,,,
1. Low,385,24,361,6.2
2. Medium,135,30,105,22.2
3. High,41,21,20,51.2


# Экспорт аналитической витрины для Tableau

In [7]:
df_clean.to_csv("data/avito_hr_datamart.csv", index=False)

In [9]:
imp_sorted = importances[importances > 0].sort_values(ascending=False)

tree_ = tree.tree_
auto_thresholds = {}
for feat_idx, thresh in zip(tree_.feature, tree_.threshold):
    if feat_idx != -2:
        fname = sig_features[feat_idx]
        if fname not in auto_thresholds:
            auto_thresholds[fname] = round(thresh, 1)

rows = []
for col, imp in imp_sorted.items():
    thresh = auto_thresholds.get(col, round(top_df[col].median(), 1))
    
    rate_le = top_df[top_df[col] <= thresh]["is_attrition"].mean()
    rate_gt = top_df[top_df[col] > thresh]["is_attrition"].mean()
    
    if rate_le >= rate_gt:
        is_risk = df_clean[col] <= thresh
        label = f"{col} <= {thresh}"
    else:
        is_risk = df_clean[col] > thresh
        label = f"{col} > {thresh}"
        
    temp = pd.DataFrame({
        "EmployeeNumber": df_clean["EmployeeNumber"],
        "Segment": df_clean["Segment"],
        "is_attrition": df_clean["is_attrition"],
        "Feature_Importance": round(imp, 4),
        "Feature_Label": label,
        "Group": np.where(is_risk, "At Risk", "Normal")
    })
    rows.append(temp)

pd.concat(rows, ignore_index=True).to_csv("data/avito_hr_dynamic_drivers.csv", index=False)

# Выводы
* По критерию Спирмена (p < 0.05) и дереву решений в топ драйверов оттока у сегмента Top Talent вошли: наличие опционов (`StockOptionLevel`), грейд (`JobLevel`), оклад (`MonthlyIncome`), удаленность от офиса (`DistanceFromHome`) и вовлеченность (`JobInvolvement`).
* С увеличением скоринга фактическая текучесть среди топов растет с 6.2% (Low Risk) до 51.2% (High Risk).
* В критическую группу попали 20 действующих топов — их данные сохранены в `data/avito_hr_datamart.csv` для дальнейшего разбора в Tableau.
